# Сопоставление подписантов JSON ↔ CSV
N-граммы сохранены; добавлена проверка порядка ФИО и расстояние Левенштейна. Совпадение с одной OCR-заменой может получить статус «Требуется проверка», а не автоматическое подтверждение. CSV разделён запятыми.

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd

# ===== НАСТРОЙКИ =====
JSON_DIR = Path("input_json")
CSV_FILE = "signers.csv"
OUTPUT_FILE = "result.xlsx"
JSON_KEYS = ["signer1", "signer2"]
CSV_COLUMN = "ФИО"
CSV_SEPARATOR = ","
SURNAME_THRESHOLD = 0.75  # Порог N-грамм
LEV_THRESHOLD = 0.85      # Дополнительный критерий для OCR-ошибок


def parse_fio(text):
    """Исходная функция: Фамилия И.О. / Фамилия Имя Отчество."""
    text = str(text).strip().lower().replace("ё", "е")
    parts = re.findall(r"[а-яa-z]+", text)
    if not parts:
        return "", ""
    return parts[0], "".join(part[0] for part in parts[1:3])


def parse_fio_auto(text):
    """Также поддерживает И.О. Фамилия и И О Фамилия."""
    text = str(text).strip().lower().replace("ё", "е")
    parts = re.findall(r"[а-яa-z]+", text)
    if len(parts) >= 3 and len(parts[0]) == len(parts[1]) == 1 and len(parts[2]) > 1:
        return parts[2], parts[0] + parts[1]
    return parse_fio(text)


def ngrams(text, n):
    if len(text) < n:
        return {text} if text else set()
    return {text[i:i+n] for i in range(len(text)-n+1)}


def ngram_similarity(a, b):
    if not a or not b:
        return 0
    if a == b:
        return 1
    scores = []
    for n in (2, 3):
        x, y = ngrams(a, n), ngrams(b, n)
        if x and y:
            scores.append(len(x & y) / len(x | y))
    return sum(scores) / len(scores) if scores else 0


def levenshtein_similarity(a, b):
    """Нормированное сходство Левенштейна без сторонних библиотек."""
    if not a or not b:
        return 0
    prev = list(range(len(b) + 1))
    for i, char_a in enumerate(a, 1):
        curr = [i]
        for j, char_b in enumerate(b, 1):
            curr.append(min(curr[-1] + 1, prev[j] + 1, prev[j-1] + (char_a != char_b)))
        prev = curr
    return 1 - prev[-1] / max(len(a), len(b))


def compare_fio(source, candidate):
    surname1, initials1 = parse_fio_auto(source)
    surname2, initials2 = parse_fio_auto(candidate)
    if not surname1 or not surname2 or len(initials1) != 2 or initials1 != initials2:
        return None
    ng = ngram_similarity(surname1, surname2)
    lev = levenshtein_similarity(surname1, surname2)
    if ng >= SURNAME_THRESHOLD:
        status = "Совпадение найдено"
    elif lev >= LEV_THRESHOLD:
        status = "Требуется проверка"
    else:
        return None
    return status, ng, lev


df = pd.read_csv(CSV_FILE, sep=CSV_SEPARATOR, encoding="utf-8-sig", dtype=str)
if CSV_COLUMN not in df.columns:
    raise ValueError(f"Нет столбца {CSV_COLUMN!r}. Доступные: {list(df.columns)}")
signers = [s.strip() for s in df[CSV_COLUMN].dropna().astype(str) if s.strip()]
if not signers:
    raise ValueError("В указанном столбце CSV нет ФИО")

results = []
for file in sorted(JSON_DIR.glob("*.json")):
    with open(file, encoding="utf-8") as f:
        data = json.load(f)
    for key in JSON_KEYS:
        original = data.get(key)
        if not isinstance(original, str) or not original.strip():
            continue
        candidates = []
        for signer in signers:
            outcome = compare_fio(original, signer)
            if outcome:
                status, ng, lev = outcome
                candidates.append((signer, status, ng, lev))
        if candidates:
            # Сначала подтвержденные совпадения, затем OCR-кандидаты; далее Левенштейн и N-граммы.
            best_name, status, ng, lev = max(
                candidates,
                key=lambda item: (item[1] == "Совпадение найдено", item[3], item[2])
            )
        else:
            best_name, status, ng, lev = "", "Совпадение не найдено", 0, 0
        results.append({
            "Файл": file.name,
            "Ключ JSON": key,
            "Исходное ФИО": original,
            "Лучшее совпадение CSV": best_name,
            "Сходство N-грамм, %": round(ng * 100, 2),
            "Сходство Левенштейна, %": round(lev * 100, 2),
            "Статус": status,
        })

columns = ["Файл", "Ключ JSON", "Исходное ФИО", "Лучшее совпадение CSV", "Сходство N-грамм, %", "Сходство Левенштейна, %", "Статус"]
pd.DataFrame(results, columns=columns).to_excel(OUTPUT_FILE, index=False)
print(f"Готово! Обработано записей: {len(results)}. Результат: {OUTPUT_FILE}")
